# Krok 3: wczytanie tekstu ze źródeł

Notebook wczytuje dwa przygotowane materiały: poradnik PDF Rzecznika Finansowego oraz zapis tekstowy strony o odszkodowaniach po wichurach. Z PDF-u wyodrębnia tekst osobno dla każdej strony; plik TXT wczytuje jako jeden dokument.

Najpierw sprawdzamy, czy ekstrakcja się powiodła i czy polskie znaki są czytelne. Następnie czyścimy tekst, dzielimy go na fragmenty, budujemy proste wyszukiwanie TF-IDF i oceniamy wyniki na przykładowych pytaniach.

**Wymagane biblioteki:** pypdf, pandas i scikit-learn. Jeśli którejś brakuje, zainstaluj ją w aktywnym środowisku Conda poleceniem python -m pip install pypdf scikit-learn.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Ścieżki do plików źródłowych

Notebook zakłada, że bieżącym katalogiem Jupytera jest folder `projekt_3_wyszukiwarka_ubezpieczenia`, w którym znajdują się oba pliki źródłowe.

In [ ]:
pdf_name = "zrodlo_poradnik_ubezpieczenia-mieszkaniowe-1.pdf"
txt_name = "zrodlo_rf_gov_pl_odszkodowania_po_wichurach.txt"

pdf_path = Path(pdf_name)
txt_path = Path(txt_name)

print(f"PDF: {pdf_path}")
print(f"TXT: {txt_path}")

## Ekstrakcja tekstu z PDF-a

`pypdf` odczyta tekst każdej strony osobno. Zachowujemy numer strony, aby później można było wskazać źródło znalezionego fragmentu.

In [ ]:
pdf_reader = PdfReader(str(pdf_path))

source_descriptions = {
    0: "Ubezpieczenia mieszkaniowe. Jak się dobrze ubezpieczyć?",
    1: "Odszkodowania po wichurach w pytaniach i odpowiedziach",
}

pdf_records = []
for page_number, page in enumerate(pdf_reader.pages, start=1):
    page_text = page.extract_text() or ""
    pdf_records.append({
        "source_id": "RF-HOME-2026",
        "source_title": 0,
        "page": page_number,
        "text": page_text,
    })

print(f"Liczba stron w PDF: {len(pdf_reader.pages)}")
print(f"Liczba stron z wyodrębnionym tekstem: {sum(bool(row['text']) for row in pdf_records)}")

## Wczytanie tekstu ze strony RF

Zapis strony ma kodowanie UTF-8. Czytamy go bez dzielenia na fragmenty — porządkowanie akapitów i przygotowanie fragmentów będzie osobnym krokiem.

In [ ]:
web_text = txt_path.read_text(encoding="utf-8")
web_text = web_text.replace("\r\n", "\n").replace("\r", "\n").strip()

web_record = {
    "source_id": "RF-STORM-2022",
    "source_title": 1,
    "page": None,
    "text": web_text,
}
print(f"Liczba znaków w pliku tekstowym: {len(web_text):,}")

## Wspólna tabela tekstów

Każdy wiersz tabeli reprezentuje stronę PDF albo cały dokument tekstowy. To prosty format roboczy, z którego będzie można skorzystać w kolejnym kroku.

In [ ]:
text_df = pd.DataFrame(pdf_records + [web_record])

text_df["char_count"] = text_df["text"].str.len()
text_df[["source_id", "source_title", "page", "char_count"]].head(10)

## Kontrola jakości ekstrakcji

Sprawdzamy puste strony i znak zastępczy `�`, który może oznaczać problem z odczytem kodowania lub tekstu. Następnie wyświetlamy przykładowy tekst z obu źródeł, żeby można było ocenić, czy akapity i polskie znaki wyglądają poprawnie.

In [ ]:
quality_df = text_df.assign(
    is_empty=text_df["text"].str.strip().eq(""),
    replacement_chars=text_df["text"].str.count("�"),
)

quality_df.groupby("source_id").agg(
    records=("text", "size"),
    empty_records=("is_empty", "sum"),
    replacement_chars=("replacement_chars", "sum"),
    total_characters=("char_count", "sum"),
)

In [ ]:
print("Przykładowy tekst z pierwszej strony PDF:\n")
print(text_df.loc[text_df["source_id"] == "RF-HOME-2026", "text"].iloc[0][:1500])

print("\n\nPoczątek tekstu strony o szkodach po wichurach:\n")
print(web_text[:2000])

## Wynik kroku 3

Tekst z obu źródeł znajduje się w `text_df`. Dla poradnika PDF zachowany jest numer strony, a materiał tekstowy jest przypisany do własnego `source_id`.

Przejrzyj przykładowy tekst i tabelę kontroli jakości. Następne komórki realizują krok 4: czyszczenie tekstu i podział na fragmenty.

# Krok 4: czyszczenie i podział tekstu na fragmenty

Usuwamy zbędne białe znaki, nagłówek z adresem źródłowym i samotne numery stron. Łączymy też wyrazy rozdzielone łącznikiem na końcu wiersza. Następnie dzielimy tekst na fragmenty o długości około 900 znaków z niewielkim nakładaniem, aby sąsiednie fragmenty zachowały część wspólnego kontekstu.

Do każdego fragmentu zachowujemy identyfikator źródła, binarną etykietę source_title (0 lub 1), numer strony (dla PDF) oraz kolejny numer fragmentu. Opisy etykiet są w zmiennej source_descriptions.

In [ ]:
import unicodedata

def clean_text(text, source_title):
    """Usuwa kilka typowych elementów technicznych i porządkuje odstępy."""
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("\xa0", " ").replace("\xad", "")
    text = re.sub(r"(?im)^Strona źródłowa:.*$", " ", text)
    text = text.replace(source_title, " ")
    text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)
    text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_into_chunks(text, chunk_size=900, overlap=120):
    """Dzieli tekst w pobliżu granicy słowa i zachowuje mały wspólny fragment."""
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            space_position = text.rfind(" ", start, end)
            if space_position > start:
                end = space_position

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)
        while start < len(text) and text[start].isspace():
            start += 1

    return chunks

In [ ]:
chunk_records = []

for _, row in text_df.iterrows():
    cleaned_text = clean_text(row["text"], source_descriptions[row["source_title"]])
    text_chunks = split_into_chunks(cleaned_text)

    for chunk_number, chunk in enumerate(text_chunks, start=1):
        chunk_records.append({
            "source_id": row["source_id"],
            "source_title": row["source_title"],
            "page": row["page"],
            "chunk_number": chunk_number,
            "text": chunk,
        })

chunks_df = pd.DataFrame(chunk_records)
chunks_df["char_count"] = chunks_df["text"].str.len()

print(f"Liczba przygotowanych fragmentów: {len(chunks_df)}")
chunks_df[["source_id", "page", "chunk_number", "char_count", "text"]].head(10)

## Podsumowanie

Obiekt `chunks_df` zawiera oczyszczone fragmenty, gotowe do indeksowania. Kolumna `source_title` ma wartości 0 lub 1, a ich pełne opisy znajdują się w `source_descriptions`. Fragmenty z poradnika PDF zachowują numer strony.

# Krok 5: wyszukiwanie fragmentów za pomocą TF-IDF

TF-IDF zamienia tekst fragmentów i pytanie na wektory liczbowe. Następnie podobieństwo cosinusowe porównuje pytanie z każdym fragmentem. Pokażemy kilka fragmentów z najwyższym wynikiem oraz opis źródła.

Wynik podobieństwa pomaga uszeregować teksty, ale nie jest prawdopodobieństwem poprawnej odpowiedzi.

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunks_df["text"])

print(f"Liczba fragmentów w indeksie: {tfidf_matrix.shape[0]}")
print(f"Liczba cech tekstowych: {tfidf_matrix.shape[1]}")

In [ ]:
def search_answers(question, top_n=5):
    """Zwraca pasujące fragmenty w kolejności od najwyższego wyniku."""
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, tfidf_matrix).flatten()

    results = chunks_df.copy()
    results["similarity"] = scores
    results["source_description"] = results["source_title"].map(source_descriptions)
    results = results[results["similarity"] > 0]
    results = results.sort_values("similarity", ascending=False)

    columns = [
        "similarity",
        "source_title",
        "source_description",
        "page",
        "chunk_number",
        "text",
    ]
    return results[columns].head(top_n).reset_index(drop=True)


## Przykładowe pytanie

Zmień tekst pytania i ponownie uruchom komórkę, aby zobaczyć najlepiej dopasowane fragmenty. W kolumnie source_title zobaczysz etykietę 0 lub 1; obok znajduje się opis źródła.

In [ ]:
question = "Co oznacza niedoubezpieczenie domu?"
search_answers(question, top_n=5)

## Sprawdzenie pytania bez dobrego dopasowania

Jeśli wyszukiwarka nie znajdzie wspólnych słów z materiałami, zwróci pustą tabelę. To prosta kontrola poza zakresem dostępnych źródeł.

In [ ]:
search_answers("Ile kosztuje składka OC samochodu?")

# Krok 6: sprawdzenie jakości wyników wyszukiwania

Sprawdzimy kilka pytań z zakresu projektu i jedno pytanie spoza zakresu. Dla każdego pytania zapisujemy, z którego źródła powinna pochodzić odpowiedź. Prosty wskaźnik Hit@3 sprawdza, czy oczekiwane źródło pojawia się wśród trzech pierwszych wyników.

To tylko wstępna kontrola: wyszukanie właściwego źródła nie gwarantuje, że zwrócony fragment w pełni odpowiada na pytanie. Dlatego trzeba też przeczytać fragmenty i ocenić je ręcznie.

In [ ]:
evaluation_questions = [
    {"question": "Co oznacza suma ubezpieczenia?", "expected_source_title": 0},
    {"question": "Czym różni się niedoubezpieczenie od nadubezpieczenia?", "expected_source_title": 0},
    {"question": "Czy powódź zawsze jest objęta ubezpieczeniem domu?", "expected_source_title": 0},
    {"question": "Co zrobić bezpośrednio po szkodzie spowodowanej wichurą?", "expected_source_title": 1},
    {"question": "Czy muszę czekać na rzeczoznawcę przed rozpoczęciem napraw?", "expected_source_title": 1},
    {"question": "Co zrobić, jeśli nie zgadzam się z wyceną szkody?", "expected_source_title": 1},
    {"question": "Czy wyszukiwarka wyliczy składkę OC samochodu?", "expected_source_title": None},
]

evaluation_rows = []
for case in evaluation_questions:
    results = search_answers(case["question"], top_n=3)
    returned_sources = results["source_title"].tolist()
    expected_source = case["expected_source_title"]

    if expected_source is None:
        source_hit = None  # pytanie spoza zakresu oceniamy ręcznie
    else:
        source_hit = expected_source in returned_sources

    if results.empty:
        top3_preview = ""
        top_similarity = None
    else:
        top3_preview = "\n\n".join(
            f"Źródło {row['source_title']} ({source_descriptions[row['source_title']]}), "
            f"strona {row['page']}: {row['text'][:250]}"
            for _, row in results.iterrows()
        )
        top_similarity = results.iloc[0]["similarity"]

    evaluation_rows.append({
        "question": case["question"],
        "expected_source_title": expected_source,
        "returned_source_titles": returned_sources,
        "expected_source_in_top3": source_hit,
        "top_similarity": top_similarity,
        "top3_preview": top3_preview,
    })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df

## Podsumowanie kontroli

Hit@3 to odsetek pytań z zakresu, dla których oczekiwane źródło pojawiło się w pierwszych trzech wynikach. Wartość obliczamy tylko dla pytań z oczekiwanym źródłem. Pytanie spoza zakresu sprawdzamy ręcznie: czy zwrócone fragmenty są nieprzydatne i czy aplikacja powinna zgłosić brak dobrego dopasowania.

Przejrzyj kolumny top_result_preview i returned_source_titles. Zgodność źródła jest tylko przybliżoną miarą — sprawdź, czy sam fragment rzeczywiście odpowiada na pytanie. Jeśli wyniki dla pytania spoza zakresu wyglądają przekonująco, ale są nie na temat, to ograniczenie prostego TF-IDF, które warto opisać w projekcie.

In [ ]:
in_scope_results = evaluation_df[evaluation_df["expected_source_title"].notna()]
hit_at_3 = in_scope_results["expected_source_in_top3"].astype(int).mean()
print(f"Hit@3 dla pytań z zakresu: {hit_at_3:.1%}")